# Semana 2 — A Geometria dos Dados: Linear Algebra com NumPy
**Fase 1: Math Foundations e ML Core** · Notebook de exercícios

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/ai-architect-foundations/blob/main/01-Phase-1-Math-and-ML-Core/02-linear-algebra-numpy/notebooks/semana_02_exercicio.ipynb)

Leitura de conceito desta semana: [Semana-02-Conceitos.md](https://github.com/NetoRibeiro/ai-architect-foundations/blob/main/01-Phase-1-Math-and-ML-Core/02-linear-algebra-numpy/docs/Semana-02-Conceitos.md)

Complete cada trecho marcado com `# [SEU CÓDIGO AQUI]`. As células seguintes dependem das anteriores: execute em ordem. O gabarito está em `semana_02_gabarito.ipynb`.

## Passo 1 — Só NumPy

In [ ]:
import numpy as np, time
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

## Passo 2 — Dot product e norma na mão
Implemente sem usar `np.dot`, `@` ou `np.linalg.norm`: use apenas soma e multiplicação elemento a elemento.

In [ ]:
def produto_escalar(a, b):
    # [SEU CÓDIGO AQUI] some os produtos elemento a elemento

def norma(a):
    # [SEU CÓDIGO AQUI] raiz do dot product de a com ele mesmo

a, b = np.array([3.0, 4.0]), np.array([4.0, 3.0])
assert produto_escalar(a, b) == 24.0 and norma(a) == 5.0
print("ok:", produto_escalar(a, b), norma(a))

## Passo 3 — Cosine similarity do zero

In [ ]:
def cosseno(a, b):
    # [SEU CÓDIGO AQUI] dot product dividido pelo produto das normas

print("mesma direção :", round(cosseno(np.array([1., 2.]), np.array([2., 4.])), 3))
print("ortogonais    :", round(cosseno(np.array([1., 0.]), np.array([0., 1.])), 3))
print("opostos       :", round(cosseno(np.array([1., 2.]), np.array([-1., -2.])), 3))
assert np.isclose(cosseno(a, b), np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

## Passo 4 — Embeddings de brinquedo
Cada "documento" é descrito por 4 características escolhidas à mão: `[finanças, tecnologia, esporte, culinária]`. É um embedding interpretável.

In [ ]:
titulos = ["Balanço trimestral do banco", "Novo chip para IA", "Final do campeonato", "Receita de risoto",
           "Fintech lança app de pagamentos", "Nutrição para maratonistas", "Custo de GPUs no data center"]
E = np.array([[.9, .1, 0, 0], [.1, .9, 0, 0], [0, .05, .95, 0], [0, 0, 0, 1],
              [.6, .7, 0, 0], [0, 0, .6, .6], [.5, .8, 0, 0]], dtype=float)
consulta = np.array([.7, .6, 0, 0])   # "investimentos em tecnologia"

## Passo 5 — Mini vector database com loop
Percorra as linhas de `E`, calcule a cosine similarity com a query e devolva os índices dos `k` mais similares, do mais para o menos similar.

In [ ]:
def buscar_loop(E, q, k=3):
    # [SEU CÓDIGO AQUI] devolva lista de (indice, score) dos k melhores

for i, s in buscar_loop(E, consulta):
    print(f"{s:.3f}  {titulos[i]}")

## Passo 6 — Versão vectorized: uma multiplicação de matriz
1. Normalize as linhas de `E` e a query (norma 1).
2. `scores = E_norm @ q_norm`.
3. Use `np.argsort` para pegar o top-k.

In [ ]:
def normalizar(M):
    # [SEU CÓDIGO AQUI] divida cada linha pela sua norma (use axis=-1, keepdims=True)

def buscar_vetorizado(E_norm, q_norm, k=3):
    # [SEU CÓDIGO AQUI] scores via @, depois argsort decrescente

E_norm, q_norm = normalizar(E), normalizar(consulta)
idx, sc = buscar_vetorizado(E_norm, q_norm)
assert list(idx) == [i for i, _ in buscar_loop(E, consulta)]
print([titulos[i] for i in idx], sc.round(3))

## Passo 7 — Escala: loop vs. matriz
50.000 documentos, 384 dimensões (tamanho típico de um modelo de embeddings pequeno).

In [ ]:
N, d = 50_000, 384
base = normalizar(rng.standard_normal((N, d)).astype(np.float32))
q = normalizar(rng.standard_normal(d).astype(np.float32))

t0 = time.perf_counter(); melhor_loop = max(range(2000), key=lambda i: cosseno(base[i], q)); t_loop = (time.perf_counter() - t0) * N / 2000
t0 = time.perf_counter(); idx, _ = buscar_vetorizado(base, q, k=5); t_vec = time.perf_counter() - t0
print(f"loop (estimado para N): {t_loop*1000:.0f} ms | vetorizado: {t_vec*1000:.1f} ms | ganho ≈ {t_loop/t_vec:.0f}x")

## Passo 8 — Normalizados: cosine, dot product e L2 dão o mesmo ranking
Verifique numericamente que `‖a − b‖² = 2 − 2(a·b)` para vetores unitários.

In [ ]:
# [SEU CÓDIGO AQUI] calcule dist2 (distância euclidiana ao quadrado) e dot para as 1000 primeiras linhas
assert np.allclose(dist2, 2 - 2 * dot, atol=1e-4)
assert (np.argsort(dist2)[:10] == np.argsort(-dot)[:10]).all()
print("Identidade confirmada: mesmo ranking.")

## Passo 9 — Matriz como transformação
Uma matriz 2×2 gira e estica o plano. Aplique `W` aos pontos de um quadrado e observe.

In [ ]:
quadrado = np.array([[0, 0], [1, 0], [1, 1], [0, 1], [0, 0]], dtype=float)
theta = np.pi / 6
W = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]]) @ np.array([[2, 0], [0, .5]])
# [SEU CÓDIGO AQUI] aplique W a cada ponto (linha) de quadrado
plt.plot(*quadrado.T, label="original"); plt.plot(*transformado.T, label="W aplicado"); plt.axis("equal"); plt.legend(); plt.show()

## Passo 10 — Maldição da dimensionalidade
Gere 500 vetores aleatórios em dimensões crescentes e veja a distribuição da cosine similarity entre pares.

In [ ]:
for dim in (2, 10, 100, 1000):
    V = normalizar(rng.standard_normal((500, dim)))
    cos = (V @ V.T)[np.triu_indices(500, 1)]
    plt.hist(cos, bins=60, alpha=.6, label=f"d={dim}", density=True)
plt.legend(); plt.xlabel("cosseno entre pares aleatórios"); plt.show()

**Reflexão:** em alta dimensão, vetores aleatórios são quase ortogonais. O que isso diz sobre um score de 0,35 entre duas frases em um embedding de 768 dimensões?